In [5]:
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd().parent  # one level up from Silver
INPUT_DIR = PROJECT / "Bronze"
OUTPUT_DIR = Path.cwd()  # Silver itself, since the script runs from here

df = pd.read_csv(INPUT_DIR / "new_car_registrations_SE.csv", encoding="latin-1")

# Keep only region-level rows (two-digit code), excluding "00 Sweden" (country total).
# Two-digit codes are regions (e.g. "01 Stockholm county"),
# four-digit codes are municipalities (e.g. "0114 Upplands Väsby").
# Country total can be derived by summing all regions - keeping it would double-count.
code_length = df["region"].str.split(" ", n=1).str[0].str.len()
df_summary = df[(code_length == 2) & (df["region"] != "00 Sweden")].copy()

# Strip the numeric code and ' county' suffix: "01 Stockholm county" -> "Stockholm"
df_summary["region"] = df_summary["region"].str.split(" ", n=1).str[1].str.replace(" county", "", regex=False)

# Map source fuel values to standardized silver-level categories
DRIVING_POWER_MAP = {
    "petrol": "petrol",
    "diesel": "diesel",
    "electricity": "electricity",
    "electric hybrid": "electric hybrid",
    "plug-in hybrid": "plug-in hybrid",
    "ethanol/ethanol flexifuel": "petrol/ethanol",
    "gas/gas flex": "gas/gas flex",
    "other fuels": "other",
}
df_summary["driving_power"] = df_summary["fuel"].map(DRIVING_POWER_MAP)

# Split the month column: "2006M01" -> year=2006, month=01
# Keep only rows starting 01/2016
df_summary["year"] = df_summary["month"].astype(str).str[:4].astype("Int64")
df_summary["month"] = df_summary["month"].astype(str).str[-2:].astype("Int64")
df_summary = df_summary[df_summary["year"] >= 2016].copy()

df_summary["country"] = "Sweden"

df_lookup = df_summary.rename(columns={
    "New registered passenger cars, number": "number_of_new_registrations",
})[[
    "country",
    "year",
    "month",
    "region",
    "number_of_new_registrations",
    "driving_power",
    "fetch_timestamp",
]]

# Convert types for consistency before saving
df_lookup["number_of_new_registrations"] = df_lookup["number_of_new_registrations"].astype(int)
df_lookup["fetch_timestamp"] = pd.to_datetime(df_lookup["fetch_timestamp"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_lookup.to_csv(OUTPUT_DIR / "silver_new_car_registrations_totals_lookup_SE.csv", index=False)